# 05 · Tool calling, giving the model hands

**AI Fundamentals in 3 Hours** · Data Sense

RAG gave the model knowledge. It still cannot **do** anything: it cannot check today's date,
look up an order, or send an email.

The one idea in this notebook, and please hold on to it:

> **The model never runs your code. It sends you a message asking you to run it.**

You decide whether to. That gate is where every safety control in agentic systems lives.

In this notebook:

1. Watch the model fail at something it cannot know
2. Turn an ordinary Python function into a tool with one decorator
3. See the `tool_calls` object the model sends back
4. Complete the round trip by hand, step by step
5. Give it several tools and watch it choose, including choosing none


In [ ]:
# --- run this first, in every notebook ---
import os, json
from pathlib import Path

# read keys out of .env (works from the repo root or from notebooks/)
for candidate in [Path(".env"), Path("../.env")]:
    if candidate.exists():
        for line in candidate.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break

from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1-mini"          # provider:model - change this one string to switch providers
model = init_chat_model(MODEL, temperature=0)

import textwrap
def wrap(text, width=88):
    """Print long text wrapped, so answers stay readable on a projector."""
    print(textwrap.fill(str(text), width=width))

assert os.environ.get("OPENAI_API_KEY"), "No API key found - check your .env file"
print("ready |", MODEL)

## 1. The failure

Ask about a specific order. There is no possible way for the model to know.

In [ ]:
wrap(model.invoke("What's the status of order 48213?").content)

It either refuses or invents something. Neither is useful.

## 2. Our fake back office

In a real system these would hit your database and your courier's API. Here they are plain
Python dictionaries so nothing is hidden.

In [ ]:
ORDERS = {
    "48213": {"status": "shipped",    "item": "Table lamp",   "total_inr": 2499,
              "courier": "Delhivery", "eta": "2026-09-16", "placed": "2026-09-08"},
    "91204": {"status": "delivered",  "item": "Storage bins", "total_inr": 4150,
              "courier": "BlueDart",  "eta": "2026-09-10", "placed": "2026-09-05"},
    "77001": {"status": "processing", "item": "Cotton throw", "total_inr": 1280,
              "courier": None,        "eta": None,          "placed": "2026-09-11"},
}
print(len(ORDERS), "orders in our pretend database")

## 3. The `@tool` decorator

This is the whole of tool definition in LangChain. The decorator reads your **function name**,
your **type hints** and your **docstring**, and builds the JSON Schema the model needs.

The docstring is not documentation. It is the text the model reads to decide whether this is
the right tool. Write it for a new colleague on their first day.

In [ ]:
from langchain_core.tools import tool

@tool
def get_order_status(order_id: str) -> dict:
    """Look up the live status, item, total and delivery ETA of a customer order.

    Use whenever the user asks where their order is or what they bought.
    Requires the numeric order id - ask the user for it if they have not given one.

    Args:
        order_id: The numeric order id, for example 48213
    """
    order = ORDERS.get(order_id.strip())
    if not order:
        return {"error": f"No order found with id {order_id}"}
    return {"order_id": order_id, **order}


@tool
def days_until(date_iso: str) -> dict:
    """Calculate how many whole days from today until a given date.

    Use for any "how many days" question. Do NOT do date arithmetic yourself.

    Args:
        date_iso: Target date as YYYY-MM-DD
    """
    from datetime import date
    try:
        target = date.fromisoformat(date_iso)
    except ValueError:
        # Return the error as DATA. Raising here would kill the whole agent run;
        # handing it back lets the model notice and correct itself next turn.
        return {"error": f"{date_iso!r} is not a date. Expected YYYY-MM-DD."}
    return {"date": date_iso, "days_from_today": (target - date.today()).days}


print("name       :", get_order_status.name)
print("description:", get_order_status.description.splitlines()[0])
print("args schema:", json.dumps(get_order_status.args_schema.model_json_schema()["properties"]))

### What the model actually receives

That decorator generated a JSON Schema for you. It is worth seeing once, so you know the
framework is doing bookkeeping rather than something mysterious.

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

schema = json.dumps(convert_to_openai_tool(get_order_status), indent=2)
for line in schema.splitlines():
    print(textwrap.fill(line, 86, subsequent_indent="      "))

Writing that by hand for every tool, keeping it in sync with the function signature, is
exactly the kind of tedium a framework should remove.

## 4. Offering the tools to the model

`bind_tools` returns a **new model** that knows about your tools. It does not call anything, it just puts the menu on the table.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

TOOLS = [get_order_status, days_until]
model_with_tools = model.bind_tools(TOOLS)

messages = [
    SystemMessage("You are a support agent for Nimbus Retail. Be concise."),
    HumanMessage("What's the status of order 48213?"),
]

reply = model_with_tools.invoke(messages)

print("content    :", repr(reply.content))
print("tool_calls :")
for c in reply.tool_calls:
    print(f"   name : {c['name']}")
    print(f"   args : {c['args']}")
    print(f"   id   : {c['id']}")
print()
print("finish_reason:", reply.response_metadata.get("finish_reason"))

### Read that carefully

`content` is empty. **It did not answer.** It asked.

Look at the `tool_calls` entry, and notice `args` is already a **Python dict**. With a raw
provider SDK you get a JSON *string* you must remember to parse, with escaping that differs
between models. LangChain has done that for you.

In [ ]:
call = reply.tool_calls[0]

print("name :", call["name"])
print("args :", call["args"], " <- a", type(call["args"]).__name__, "not a string")
print("id   :", call["id"])

## 5. *You* execute the tool

This is the step everybody gets wrong. Nothing automatic happens here. You look at what was
requested, you decide whether to allow it, and you run it yourself.

Passing the whole tool call to `.invoke()` gives you back a ready-made `ToolMessage`, with
the `tool_call_id` already wired up.

In [ ]:
tool_message = get_order_status.invoke(call)     # in production: check the name against an allowlist

print(type(tool_message).__name__)
print("content      :", tool_message.content[:70], "...")
print("tool_call_id :", tool_message.tool_call_id)

## 6. Send it back

The model is stateless, remember. It has no memory of asking. You resend everything, the
original messages, the assistant message containing the tool call, **and** the result.

In [ ]:
messages.append(reply)            # the assistant's tool-call message
messages.append(tool_message)     # the result

final = model_with_tools.invoke(messages)
wrap(final.content)

**That is tool calling.** Two API round trips for one tool use, which is exactly why
agents are slower and more expensive than a single call.

Here is the shape of the conversation now:

In [ ]:
for m in messages + [final]:
    label = type(m).__name__
    if getattr(m, "tool_calls", None):
        tc = m.tool_calls[0]
        print(f"{label:<14}| TOOL CALL {tc['name']}({tc['args']})")
    else:
        print(f"{label:<14}| {str(m.content)[:70]}")

## 7. Wrap it up

The reusable version. Note it handles **several tool calls in one reply:** models routinely
request two or three at once, and all the results must go back before the next call.

In [ ]:
BY_NAME = {t.name: t for t in TOOLS}

def run_with_tools(question: str, verbose: bool = True) -> str:
    msgs = [
        SystemMessage("You are a support agent for Nimbus Retail. Be concise."),
        HumanMessage(question),
    ]

    reply = model_with_tools.invoke(msgs)
    msgs.append(reply)

    if not reply.tool_calls:
        return reply.content                      # it answered directly - no tool needed

    for call in reply.tool_calls:                 # may be more than one
        result = BY_NAME[call["name"]].invoke(call)
        if verbose:
            print(f"  [tool] {call['name']}({call['args']})")
            print(f"         -> {result.content[:64]}")
        msgs.append(result)

    return model_with_tools.invoke(msgs).content


wrap(run_with_tools("Where is order 77001?"))

## 8. Letting it choose

Give it a mix of questions and watch which tool it picks, and when it picks none at all.

In [ ]:
for q in [
    "What did I order in 91204 and how much was it?",
    "How many days until 2026-12-25?",
    "What is the capital of Karnataka?",        # no tool should be used
    "Is order 99999 shipped?",                  # tool used, order does not exist
]:
    print("=" * 74)
    print("Q:", q)
    wrap("A: " + run_with_tools(q))
    print()

Three things to notice:

1. **It picked the right tool** for the first two, without you routing anything.
2. **It used no tool** for the Karnataka question. It knows that one. Tools are offered, not forced.
3. **The missing order was handled gracefully:** our function returned `{"error": ...}` as
   *data*, the model read it and explained the problem in plain language.

That third point is a real production pattern: **return errors, don't raise them**. An
exception kills your run. A returned error gives the model a chance to recover.

## 9. The limit of one round trip

Ask something that needs two steps: look up the order, *then* count the days to its ETA.

In [ ]:
answer = run_with_tools("Order 48213 - how many days until it arrives?")

print("\nfinal answer:", repr(answer))

**The final answer is empty.** That is not a bug in your code, it is the limitation,
showing itself.

Let's look at what the model actually said on that second call.

In [ ]:
msgs = [
    SystemMessage("You are a support agent for Nimbus Retail. Be concise."),
    HumanMessage("Order 48213 - how many days until it arrives?"),
]
first = model_with_tools.invoke(msgs)
msgs += [first, BY_NAME[first.tool_calls[0]["name"]].invoke(first.tool_calls[0])]

second = model_with_tools.invoke(msgs)
print("round 1 asked for :", [c["name"] for c in first.tool_calls])
print("round 2 content   :", repr(second.content))
print("round 2 asked for :", [c["name"] for c in second.tool_calls])
print()
print("It learned the ETA, then asked for a SECOND tool - and our function")
print("had already stopped listening.")

It called `get_order_status`, learned the ETA is 2026-09-16, and then asked for
`days_until`: but `run_with_tools` only goes around **once**, so nobody answered.

That missing piece *keep going until the job is done* is the entire idea of an agent.

It is about four lines of code away, and LangChain has already written them for you.

**That is notebook 06.**

## 10. Your turn

1. **Write a tool.** Add `cancel_order(order_id)` that flips a status to `"cancelled"`. Ask
   the model to cancel one, and notice how casually it will destroy data for you. Think about
   what approval gate belongs in front of it, notebook 06 adds exactly that.

2. **Sabotage the docstring.** Change `days_until`'s docstring to just `"Date helper."` and
   rerun the date question. Watch it stop choosing the tool. The docstring *is* the interface.

3. **Force a bad argument.** Ask "check order forty-eight thousand two hundred and thirteen"
   and look at what lands in `args`. Tool arguments are model output, treat them exactly like
   untrusted user input, and validate them.


In [ ]:
# your turn - scratch cell
